# 01 – Data Exploration
**Project:** Diabetes Prediction ML  
**Seminar:** Advanced Applied Data Science – Goethe Universität Frankfurt  
**Dataset:** CDC BRFSS 2015 – Diabetes Health Indicators (UCI #891)  
**CRISP-DM Phase:** 2 – Data Understanding  

---

## Background
The Behavioral Risk Factor Surveillance System (BRFSS) is a telephone survey conducted annually by the U.S. Centers for Disease Control and Prevention (CDC). Respondents self-report health behaviors, chronic conditions, and preventive service use. This notebook explores the 2015 edition, which includes ~253,680 survey responses and a binary diabetes indicator as the target variable.

**Target variable:** `Diabetes_binary`
- `0` = No diabetes
- `1` = Prediabetes or diabetes

**Goal of this notebook:** Understand the structure, distributions, class balance, and feature relationships of the dataset before any preprocessing or modelling.

## Setup
Install the UCI ML Repository client if not already available, then import all required libraries.

In [ ]:
# Install ucimlrepo if running in a fresh environment
import importlib, subprocess, sys
if importlib.util.find_spec('ucimlrepo') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ucimlrepo'])

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr
from ucimlrepo import fetch_ucirepo

# Global plot style
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Output directory for saving plots
PLOTS_DIR = os.path.join('..', 'results', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

LABEL_MAP = {0: 'No Diabetes', 1: 'Prediabetes/Diabetes'}
CLASS_COLORS = {0: '#4CAF50', 1: '#E53935'}

print('Libraries loaded successfully.')

## 1 · Load the Dataset
We fetch the dataset directly from the UCI ML Repository using its official ID (891). This ensures reproducibility without requiring a local CSV file.

In [ ]:
# Fetch dataset from UCI ML Repository
dataset = fetch_ucirepo(id=891)

X = dataset.data.features
y = dataset.data.targets

# Combine into a single DataFrame for convenience
df = pd.concat([X, y], axis=1)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2 · Data Structure
Before any analysis, we inspect the raw structure of the dataset: shape, column types, summary statistics, and – critically – the absence of missing values.

In [ ]:
print('=== Shape ===')
print(f'{df.shape[0]:,} samples,  {df.shape[1]} features (including target)')

print('\n=== Column dtypes ===')
print(df.dtypes)

In [ ]:
# Full info output (non-null counts confirm completeness)
df.info()

In [ ]:
# Descriptive statistics for all columns
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Explicit missing-value check
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None – dataset is complete.')

**Interpretation:**  
The dataset contains exactly 253,680 rows and 22 columns (21 features + 1 target). All values are numeric; the UCI source confirms no missing values, which we verify above. The majority of features are binary (0/1) or ordinal integers. No imputation will be required in the Data Preparation phase.

## 3 · Target Variable – Class Distribution
The most fundamental property of a classification dataset is the balance between classes. We quantify the imbalance and visualise the absolute counts alongside percentages.

In [ ]:
target_col = 'Diabetes_binary'
counts = df[target_col].value_counts().sort_index()
pcts   = df[target_col].value_counts(normalize=True).sort_index() * 100

class_df = pd.DataFrame({
    'Class': [LABEL_MAP[i] for i in counts.index],
    'Count': counts.values,
    'Percent (%)': pcts.values.round(2)
})
print(class_df.to_string(index=False))

imbalance_ratio = counts[0] / counts[1]
print(f'\nImbalance ratio (majority : minority) = {imbalance_ratio:.2f} : 1')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    [LABEL_MAP[i] for i in counts.index],
    counts.values,
    color=[CLASS_COLORS[i] for i in counts.index],
    edgecolor='white',
    linewidth=1.2,
    width=0.5
)

# Annotate bars with absolute count and percentage
for bar, count, pct in zip(bars, counts.values, pcts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1500,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Class Distribution – Diabetes_binary', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Samples')
ax.set_xlabel('Class')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, counts.max() * 1.18)
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '01_class_distribution.png'), bbox_inches='tight')
plt.show()

**Interpretation:**  
The dataset is clearly imbalanced: approximately **86 %** of respondents report no diabetes, while only **14 %** report prediabetes or diabetes. The majority-to-minority ratio is roughly **6:1**. This imbalance must be addressed during Data Preparation – candidate strategies include SMOTE oversampling, class-weight adjustment in the model, or a combination of both. Accuracy alone will be a misleading metric; we should prioritise **AUC-ROC**, **F1-score**, and **Recall** (sensitivity).

## 4 · Feature Overview
The 21 features are grouped by their measurement scale:

| Type | Features |
|------|----------|
| **Binary (0/1)** | HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex |
| **Ordinal** | GenHlth (1–5), Education (1–6), Income (1–8), Age (1–13) |
| **Numeric (continuous-ish)** | BMI, MentHlth (0–30), PhysHlth (0–30) |

In [ ]:
binary_features = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]
ordinal_features = ['GenHlth', 'Education', 'Income', 'Age']
numeric_features = ['BMI', 'MentHlth', 'PhysHlth']

print(f'Binary features  : {len(binary_features)}')
print(f'Ordinal features : {len(ordinal_features)}')
print(f'Numeric features : {len(numeric_features)}')
print(f'Target           : {target_col}')
print(f'Total columns    : {len(binary_features) + len(ordinal_features) + len(numeric_features) + 1}')

### 4.1 Binary Feature Distributions
For each binary feature we show the proportion of 0 vs 1 responses across the full dataset.

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(binary_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(binary_features):
    val_counts = df[feat].value_counts(normalize=True).sort_index() * 100
    axes[i].bar(val_counts.index.astype(str), val_counts.values,
                color=['#78909C', '#42A5F5'], edgecolor='white')
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Share (%)')
    axes[i].set_ylim(0, 105)
    for j, (val, pct) in enumerate(zip(val_counts.index, val_counts.values)):
        axes[i].text(j, pct + 1.5, f'{pct:.1f}%', ha='center', fontsize=8)

# Hide any unused subplots
for idx in range(len(binary_features), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Binary Feature Distributions (% of total)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_binary_feature_distributions.png'), bbox_inches='tight')
plt.show()

### 4.2 Ordinal Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, len(ordinal_features), figsize=(16, 5))

for ax, feat in zip(axes, ordinal_features):
    val_counts = df[feat].value_counts(normalize=True).sort_index() * 100
    ax.bar(val_counts.index.astype(str), val_counts.values,
           color=sns.color_palette('muted', len(val_counts)), edgecolor='white')
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Share (%)')

fig.suptitle('Ordinal Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_ordinal_feature_distributions.png'), bbox_inches='tight')
plt.show()

### 4.3 Numeric Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, len(numeric_features), figsize=(16, 5))

for ax, feat in zip(axes, numeric_features):
    ax.hist(df[feat], bins=40, color='#5C6BC0', edgecolor='white', alpha=0.85)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_numeric_feature_distributions.png'), bbox_inches='tight')
plt.show()

**Interpretation:**  
Most binary features are skewed toward one value (e.g., CholCheck is 1 for ~96% of respondents). BMI approximates a right-skewed normal distribution centred around 28. MentHlth and PhysHlth are strongly zero-inflated – the majority of respondents report 0 bad days, with a secondary spike at 30 (maximum). This bimodal pattern suggests these features may behave more like ordinal indicators than continuous variables.

## 5 · Feature Distributions by Diabetes Status
We now compare each feature's distribution separately for the two classes. This reveals which features visually discriminate between diabetic and non-diabetic respondents.

### 5.1 Binary Features – Diabetes Rate per Group
For each binary feature we compute the share of diabetes cases within each value group (0 or 1).

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(binary_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(binary_features):
    # Group by feature value, compute mean of target (= diabetes rate)
    rate = df.groupby(feat)[target_col].mean() * 100
    bars = axes[i].bar(
        rate.index.astype(str), rate.values,
        color=['#78909C', '#EF5350'], edgecolor='white'
    )
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Diabetes rate (%)')
    axes[i].set_ylim(0, rate.max() * 1.25)
    for bar, val in zip(bars, rate.values):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=8
        )

for idx in range(len(binary_features), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Diabetes Rate by Binary Feature Value', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '05_binary_features_diabetes_rate.png'), bbox_inches='tight')
plt.show()

### 5.2 Ordinal Features – Distribution by Class

In [ ]:
fig, axes = plt.subplots(1, len(ordinal_features), figsize=(18, 5))

for ax, feat in zip(axes, ordinal_features):
    for cls in [0, 1]:
        subset = df[df[target_col] == cls][feat].value_counts(normalize=True).sort_index() * 100
        ax.plot(subset.index, subset.values, marker='o',
                label=LABEL_MAP[cls], color=CLASS_COLORS[cls], linewidth=2)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Share within class (%)')
    ax.legend(fontsize=8)

fig.suptitle('Ordinal Features – Distribution by Diabetes Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '06_ordinal_features_by_class.png'), bbox_inches='tight')
plt.show()

### 5.3 Numeric Features – KDE by Class

In [ ]:
fig, axes = plt.subplots(1, len(numeric_features), figsize=(16, 5))

for ax, feat in zip(axes, numeric_features):
    for cls in [0, 1]:
        subset = df[df[target_col] == cls][feat]
        subset.plot.kde(ax=ax, label=LABEL_MAP[cls],
                        color=CLASS_COLORS[cls], linewidth=2.5, alpha=0.85)
        ax.axvline(subset.median(), color=CLASS_COLORS[cls],
                   linestyle='--', linewidth=1, alpha=0.7)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

fig.suptitle('Numeric Features – KDE by Diabetes Status (dashed = median)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '07_numeric_features_kde_by_class.png'), bbox_inches='tight')
plt.show()

**Interpretation:**  
- **Binary features:** HighBP, HighChol, Stroke, HeartDiseaseorAttack, and DiffWalk show particularly large differences in diabetes rates between value groups – these are likely strong predictors.
- **Ordinal features:** Diabetic respondents cluster at worse health categories: lower GenHlth scores (poorer health), lower education, lower income, and older age groups.
- **BMI:** The diabetic distribution is shifted clearly to the right (higher BMI), confirming BMI as a clinically meaningful predictor. MentHlth and PhysHlth show smaller but visible separation.

## 6 · Correlation Analysis
We compute pairwise Pearson correlations across all 22 columns (21 features + target) to identify multicollinearity and feature–target relationships.

In [ ]:
corr_matrix = df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='white',
    ax=ax, cbar_kws={'shrink': 0.7}
)

ax.set_title('Pearson Correlation Matrix – All Features + Target', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '08_correlation_heatmap.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 features correlated with target
target_corr = (
    corr_matrix[target_col]
    .drop(target_col)
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E53935' if corr_matrix[target_col][f] > 0 else '#1565C0' for f in target_corr.index]
bars = ax.barh(target_corr.index[::-1], target_corr.values[::-1],
               color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, target_corr.values[::-1]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_title('Top 10 Features by |Correlation| with Diabetes_binary',
             fontsize=13, fontweight='bold')
ax.set_xlabel('|Pearson r|')
ax.set_xlim(0, target_corr.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '09_top10_feature_correlation.png'), bbox_inches='tight')
plt.show()

print('Top 10 correlations with target:')
print(target_corr.to_string())

**Interpretation:**  
As expected from domain knowledge, the strongest correlates with `Diabetes_binary` are:
1. **GenHlth** – poor self-rated health is strongly linked to diabetes
2. **HighBP** – hypertension and diabetes frequently co-occur
3. **BMI** – a well-established risk factor
4. **Age** – risk increases with age
5. **HighChol** – cholesterol problems often accompany diabetes

No pair of features is so highly correlated (r > 0.9) that it would cause severe multicollinearity for linear models. GenHlth and DiffWalk show moderate inter-feature correlation (~0.4), which is worth monitoring.

## 7 · Statistical Tests
Correlation coefficients assume linearity. We complement them with formal statistical tests:
- **Chi-Square test** for binary/categorical features vs. `Diabetes_binary` (both are categorical)
- **Point-biserial correlation** for continuous/numeric features vs. `Diabetes_binary`

In [ ]:
results = []

# Chi-Square for binary and ordinal features
for feat in binary_features + ordinal_features:
    contingency = pd.crosstab(df[feat], df[target_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    results.append({
        'Feature': feat,
        'Test': 'Chi-Square',
        'Statistic': round(chi2, 2),
        'p-value': p,
        'DoF': dof
    })

# Point-biserial for numeric features
for feat in numeric_features:
    r, p = pointbiserialr(df[target_col], df[feat])
    results.append({
        'Feature': feat,
        'Test': 'Point-biserial r',
        'Statistic': round(r, 4),
        'p-value': p,
        'DoF': '-'
    })

results_df = (
    pd.DataFrame(results)
    .sort_values('p-value')
    .reset_index(drop=True)
)

# Format p-values for display
results_df['p-value (fmt)'] = results_df['p-value'].apply(
    lambda p: f'{p:.2e}' if p < 0.001 else f'{p:.4f}'
)

display_df = results_df[['Feature', 'Test', 'Statistic', 'p-value (fmt)', 'DoF']]
display_df.style.background_gradient(cmap='Reds_r', subset=[])

In [ ]:
print('Statistical test results (sorted by p-value):')
print(display_df.to_string(index=False))

sig_count = (results_df['p-value'] < 0.05).sum()
print(f'\nFeatures significant at α=0.05: {sig_count} / {len(results_df)}')

**Interpretation:**  
All 21 features are statistically significant predictors of diabetes status (p < 0.05), which is expected given the large sample size (~254k) – even tiny effects become statistically detectable. Statistical significance alone therefore says nothing about practical importance. We must rely on effect sizes (correlation coefficients, odds ratios) and model-based feature importance to prioritise features in later phases.

## 8 · Outlier Analysis
We focus on the three numeric features (BMI, MentHlth, PhysHlth) using the IQR method. Outliers are defined as values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR.

In [ ]:
fig, axes = plt.subplots(1, len(numeric_features), figsize=(14, 6))

for ax, feat in zip(axes, numeric_features):
    ax.boxplot(
        df[feat].dropna(),
        vert=True, patch_artist=True,
        boxprops=dict(facecolor='#90CAF9', color='#1565C0'),
        medianprops=dict(color='#E53935', linewidth=2),
        flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#757575'),
        whiskerprops=dict(color='#1565C0'),
        capprops=dict(color='#1565C0')
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_ylabel('Value')

fig.suptitle('Boxplots – Numeric Features (outliers as grey dots)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '10_numeric_boxplots.png'), bbox_inches='tight')
plt.show()

In [ ]:
outlier_stats = []
for feat in numeric_features:
    q1 = df[feat].quantile(0.25)
    q3 = df[feat].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[feat] < lower) | (df[feat] > upper)).sum()
    outlier_stats.append({
        'Feature': feat,
        'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'Lower fence': round(lower, 2),
        'Upper fence': round(upper, 2),
        'N outliers': n_outliers,
        'Outlier %': round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_stats)
print(outlier_df.to_string(index=False))

**Interpretation:**  
- **BMI** has the most meaningful outliers: values above the upper fence represent extremely high BMI (likely ≥ ~52), which are medically plausible but rare. Given that BMI is a core predictor of diabetes, these should be retained rather than removed – tree-based models handle them natively, and winsorising would distort a clinically relevant range.
- **MentHlth / PhysHlth** are bounded 0–30 by definition, so outliers per IQR simply reflect the bimodal distribution (spike at 30). No action needed.
- Overall, outlier removal is not recommended for this dataset.

## 9 · Skewness & Kurtosis
We compute skewness and excess kurtosis for all numerical and ordinal features to understand the shape of their distributions. This informs feature transformation decisions in the Data Preparation phase.

In [ ]:
shape_cols = numeric_features + ordinal_features

shape_df = pd.DataFrame({
    'Feature': shape_cols,
    'Skewness': [df[c].skew() for c in shape_cols],
    'Excess Kurtosis': [df[c].kurtosis() for c in shape_cols]
}).set_index('Feature').round(3)

shape_df['Skew interpretation'] = shape_df['Skewness'].apply(
    lambda s: 'Highly right-skewed' if s > 1
    else ('Moderately right-skewed' if s > 0.5
    else ('Approximately symmetric' if abs(s) <= 0.5
    else ('Moderately left-skewed' if s > -1
    else 'Highly left-skewed')))
)

print(shape_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Skewness barplot
skew_sorted = shape_df['Skewness'].sort_values()
colors_skew = ['#EF5350' if v > 0 else '#42A5F5' for v in skew_sorted]
axes[0].barh(skew_sorted.index, skew_sorted.values, color=colors_skew, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Skewness', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Skewness value')

# Kurtosis barplot
kurt_sorted = shape_df['Excess Kurtosis'].sort_values()
colors_kurt = ['#AB47BC' if v > 0 else '#78909C' for v in kurt_sorted]
axes[1].barh(kurt_sorted.index, kurt_sorted.values, color=colors_kurt, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Excess Kurtosis', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Kurtosis value (excess)')

fig.suptitle('Distribution Shape – Skewness & Kurtosis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '11_skewness_kurtosis.png'), bbox_inches='tight')
plt.show()

**Interpretation:**
- **MentHlth and PhysHlth** are highly right-skewed and leptokurtic due to the large mass at 0. A log1p transform or binary encoding (0 vs ≥1 bad days) could be considered.
- **BMI** is moderately right-skewed; a log transform might help linear/distance-based models.
- **GenHlth and Age** are roughly symmetric. **Income and Education** are left-skewed, indicating fewer respondents in the lowest categories.
- Tree-based models (Random Forest, XGBoost) are invariant to monotone transforms, so skewness is a concern primarily for logistic regression or distance-based methods.

## 10 · Summary

### Dataset Overview
| Property | Value |
|----------|-------|
| Samples | 253,680 |
| Features | 21 |
| Target | Diabetes_binary (binary) |
| Missing values | None |
| Feature types | 14 binary, 4 ordinal, 3 numeric |

### Class Imbalance
The dataset is significantly imbalanced (~86% No Diabetes vs. ~14% Prediabetes/Diabetes), with a majority-to-minority ratio of approximately **6:1**. This must be addressed before or during modelling. Candidate approaches:
- **SMOTE** oversampling on the training set
- `class_weight='balanced'` in scikit-learn estimators
- Threshold tuning post-prediction

### Top 5 Predictors (by correlation & statistical tests)
1. **GenHlth** – general health self-rating (1=excellent, 5=poor)
2. **HighBP** – hypertension diagnosis
3. **BMI** – body mass index (continuous)
4. **Age** – age category (1–13)
5. **HighChol** – high cholesterol diagnosis

These align with established clinical risk factors for type 2 diabetes.

### Identified Data Issues
| Issue | Detail | Recommended Action |
|-------|--------|--------------------|
| Class imbalance | 6:1 ratio | SMOTE / class weights |
| BMI outliers | Values up to ~98 | Retain (clinically valid), consider winsorising for linear models |
| MentHlth / PhysHlth zero-inflation | ~75% report 0 bad days | Consider log1p transform or binary encoding |
| All features numeric (floats) | Should be int for binary/ordinal | Cast to int in preprocessing |

### Recommendations for Data Preparation (Phase 3)
1. **Handle class imbalance** using SMOTE (fit only on training data to prevent leakage)
2. **Feature scaling** for linear/distance-based models (StandardScaler or MinMaxScaler)
3. **Optional transforms** for MentHlth and PhysHlth (log1p or binarise)
4. **Train/test split** before any resampling or imputation (stratified, 80:20)
5. **Feature selection** experiments based on the top correlators identified here
6. **Primary metric:** AUC-ROC + F1-score (weighted toward minority class recall)
